In [13]:
import pandas as pd
import numpy as np
import re
import unicodedata

In [2]:
activaciones = pd.read_csv(r"C:\Users\bario\OneDrive\Documentos\UC3M\PROYECTO EN CIENCIA DE DATOS\proyecto_emergencias-20260915T155207Z-1-001\proyecto_emergencias\activaciones.csv")
activaciones.head()

,activacion_id,año,mes,hora_solicitud,hora_intervencion,tipo_emergencia,distrito,hospital,unidad_id,prioridad,paciente_edad,paciente_sexo
0,1,2024,ENERO,7:10:00,7:17:00,Intoxicación etílica,FUENCARRAL,NaN,39,alta,69.0,NaN
1,2,2024,ENERO,0:17:12,NaN,Agresión sin especificar,CENTRO,NaN,29,baja,65.0,H
2,3,2024,ENERO,0:21:25,0:30:45,Apertura de puerta,LATINA,NaN,15,alta,62.0,H
3,4,2024,ENERO,0:47:26,NaN,Patología cardiovascular,CENTRO,NaN,43,media,75.0,H
4,5,2024,ENERO,0:41:25,0:46:56,Intoxicación etílica,CIUDAD LINEAL,NaN,8,media,50.0,M


In [3]:
activaciones_limpio = activaciones.copy()

**EMPEZAMOS CON UNA ANALISIS DEL DATASET PARA VER LAS INCONSISTENCIAS EXISTENTES**
---

In [20]:
# 1. ESTRUCTURA GENERAL, NULOS Y DUPLICADOS
print("\n===== 1. ANALISIS GENERAL, NULL, DUPLICADOS =====")
resumen_estructura = pd.DataFrame({
    'Tipo': activaciones_limpio.dtypes,
    'No-Nulos': activaciones_limpio.notnull().sum(),
    'Nulos': activaciones_limpio.isnull().sum(),
    '% Nulos': (activaciones_limpio.isnull().sum() / len(activaciones_limpio) * 100).round(2)
})
print(resumen_estructura)
print(f"\nFilas duplicadas exactas: {activaciones_limpio.duplicated().sum()}")


# 2. ANÁLISIS DE COLUMNAS TEMPORALES (AÑO Y MES)
print("\n===== 2. ANÁLISIS DE COLUMNAS AÑO Y MES =====")
print(f"--- Columna 'año' (Nulos: {activaciones_limpio['año'].isnull().sum()}) ---")
print("Valores únicos detectados en 'año':", activaciones_limpio['año'].unique().tolist())

val_mes_raw = activaciones_limpio['mes'].dropna().astype(str).tolist()
con_tilde_mes = sum(1 for x in val_mes_raw if re.search(r'[áéíóúÁÉÍÓÚ]', x))
con_espacios_mes = sum(1 for x in val_mes_raw if x != x.strip())
con_minusculas_mes = sum(1 for x in val_mes_raw if not x.isupper())

print(f"\n--- Columna 'mes' (Nulos: {activaciones_limpio['mes'].isnull().sum()}) ---")
print("Valores únicos detectados en 'mes':", activaciones_limpio['mes'].unique().tolist())
print("Inconsistencias detectadas en 'mes':")
print(f"  - Con minúsculas / mezcla de caja: {con_minusculas_mes}")
print(f"  - Con tildes: {con_tilde_mes}")
print(f"  - Con espacios sobrantes: {con_espacios_mes}")


# 3. ANÁLISIS DE FORMATO EN HORAS
print("\n===== 3. COLUMNAS DE HORA =====")
for col_hora in ['hora_solicitud', 'hora_intervencion']:
    horas_str = activaciones_limpio[col_hora].dropna().astype(str)
    con_guion = horas_str.str.contains('-').sum()
    con_dos_puntos = horas_str.str.contains(':').sum()
    patron_hh_mm_ss = r'^\d{1,2}:\d{2}:\d{2}$'
    no_cumplen_formato = (~horas_str.str.match(patron_hh_mm_ss)).sum()
    
    print(f"--- Columna '{col_hora}' (Nulos: {activaciones_limpio[col_hora].isnull().sum()}) ---")
    print(f"  - Registros con guión ('-') en vez de ':' : {con_guion}")
    print(f"  - Registros con ':' estándar: {con_dos_puntos}")
    print(f"  - Formato no estándar (ej. 0-03-49): {no_cumplen_formato}")


# 4. RANGOS Y VALORES EN EDAD Y UNIDAD
print("\n===== 4. COHERENCIA DE VARIABLES NUMÉRICAS Y RANGOS =====")
columnas_numericas = ['paciente_edad', 'unidad_id']
print(activaciones_limpio[columnas_numericas].describe().T[['min', 'mean', 'max']])

edades_raras = activaciones_limpio[(activaciones_limpio['paciente_edad'] < 0) | (activaciones_limpio['paciente_edad'] > 115)]
print(f"\n- Pacientes con edad fuera de rango (0-115 años): {len(edades_raras)}")


# 5. VALORES ÚNICOS Y VARIACIONES EN CATEGÓRICAS CRÍTICAS
print("\n===== 5. VALORES ÚNICOS EN VARIABLES CATEGÓRICAS =====")
cols_categoricas = ['tipo_emergencia', 'distrito', 'hospital', 'prioridad', 'paciente_sexo']

for col in cols_categoricas:
    val_raw = activaciones_limpio[col].dropna().astype(str).tolist()
    
    con_tilde = sum(1 for x in val_raw if re.search(r'[áéíóúÁÉÍÓÚ]', x))
    con_espacios = sum(1 for x in val_raw if x != x.strip())
    con_minusculas = sum(1 for x in val_raw if not x.isupper())
    
    print(f"\n--- Columna: '{col}' (Nulos: {activaciones_limpio[col].isnull().sum()} | {(activaciones_limpio[col].isnull().sum()/len(activaciones_limpio)*100):.2f}%) ---")
    print(f"Valores únicos ({activaciones_limpio[col].nunique()} distintos): {activaciones_limpio[col].unique().tolist()[:10]}...")
    print(f"Inconsistencias detectadas:")
    print(f"  - Con minúsculas / mezcla de caja: {con_minusculas}")
    print(f"  - Con tildes: {con_tilde}")
    print(f"  - Con espacios sobrantes: {con_espacios}")


===== 1. ANALISIS GENERAL, NULL, DUPLICADOS =====
                      Tipo  No-Nulos  Nulos  % Nulos
activacion_id        int64    152844      0     0.00
año                  int64    152844      0     0.00
mes                    str    152844      0     0.00
hora_solicitud         str    151258   1586     1.04
hora_intervencion      str    138807  14037     9.18
tipo_emergencia        str    152844      0     0.00
distrito               str    152844      0     0.00
hospital               str    152844      0     0.00
unidad_id            int64    152844      0     0.00
prioridad              str    152844      0     0.00
paciente_edad      float64    151894    950     0.62
paciente_sexo          str    152844      0     0.00

Filas duplicadas exactas: 738

===== 2. ANÁLISIS DE COLUMNAS AÑO Y MES =====
--- Columna 'año' (Nulos: 0) ---
Valores únicos detectados en 'año': [2024]

--- Columna 'mes' (Nulos: 0) ---
Valores únicos detectados en 'mes': ['ENERO', 'FEBRERO', 'MARZO', 'ABRIL

### OBSERVACIONES

1. **Duplicados y Estructura**: Se registran 614 filas duplicadas exactas sobre un total de 152.844 registros. Además, se detecta una alta densidad de nulos en variables clave como `hospital` (65.61%), `hora_intervencion` (9.18%) y `paciente_sexo` (3.95%). Solución -> examinar las filas duplicadas exactas e imputar los nulos con etiquetas o descriptores adecuados según corresponda.
2. **Fragmentación Categórica en Mes**: El campo `mes` presenta 72 variantes sintácticas para representar únicamente los 12 meses del año, mezclando español, inglés, nombres completos, abreviaturas, valores numéricos de dos dígitos (`01`-`12`) y variaciones entre mayúsculas y minúsculas. Solución -> mapear con un diccionario completo para homogeneizar a nombres de mes en español, mayúsculas y sin tildes.
3. **Inconsistencias en Columnas Temporales de Hora**: En `hora_solicitud` existen 200 registros que utilizan guiones (`-`) en lugar de dos puntos (`:`) como separadores (ej. `0-03-49`), además de formatos no estandarizados que carecen de ceros a la izquierda. Solución -> corregir los separadores mediante expresiones regulares y parsear a formato de hora estandarizado `HH:MM:SS`.
4. **Anomalías de Coherencia Física en Edad**: Se detectan 150 registros con valores de `paciente_edad` fuera del rango biológico lógico (de -10.0 a 199.0 años). Solución -> tratar o imputar los valores atípicos y negativos mediante filtrado o sustitución por nulos.
5. **Heterogeneidad en Variables Categóricas (Tipo Emergencia, Distrito, Hospital y Prioridad)**: `tipo_emergencia` contiene 112 variantes con minúsculas y tildes; `distrito` y `hospital` presentan diferencias de caja, tildes y nombres abreviados (`HULP`, `HGUGM`); y `prioridad` combina números (`1`), minúsculas y términos equivalentes (`urgente`, `normal`). Solución -> estandarizar todos los textos a mayúsculas, eliminar tildes y unificar categorías equivalentes.

**SOLUCIÓN FORMATO MES**
- Estandarizar todos los meses a español mayusculas


In [7]:
# Diccionario de mapeo unificado para los 12 meses
mapeo_meses = {
    # Enero
    'ENERO': 'ENERO', 'January': 'ENERO', 'enero': 'ENERO', 'Enero': 'ENERO', '01': 'ENERO', 'ENE': 'ENERO',
    # Febrero
    'FEBRERO': 'FEBRERO', 'Febrero': 'FEBRERO', '02': 'FEBRERO', 'February': 'FEBRERO', 'febrero': 'FEBRERO', 'FEB': 'FEBRERO',
    # Marzo
    'MARZO': 'MARZO', '03': 'MARZO', 'MAR': 'MARZO', 'Marzo': 'MARZO', 'marzo': 'MARZO', 'March': 'MARZO',
    # Abril
    'ABRIL': 'ABRIL', 'ABR': 'ABRIL', '04': 'ABRIL', 'April': 'ABRIL', 'Abril': 'ABRIL', 'abril': 'ABRIL',
    # Mayo
    'MAYO': 'MAYO', '05': 'MAYO', 'May': 'MAYO', 'mayo': 'MAYO', 'Mayo': 'MAYO', 'MAY': 'MAYO',
    # Junio
    'JUNIO': 'JUNIO', 'junio': 'JUNIO', 'Junio': 'JUNIO', 'JUN': 'JUNIO', 'June': 'JUNIO', '06': 'JUNIO',
    # Julio
    'JULIO': 'JULIO', '07': 'JULIO', 'Julio': 'JULIO', 'July': 'JULIO', 'julio': 'JULIO', 'JUL': 'JULIO',
    # Agosto
    'AGOSTO': 'AGOSTO', 'Agosto': 'AGOSTO', 'AGO': 'AGOSTO', '08': 'AGOSTO', 'August': 'AGOSTO', 'agosto': 'AGOSTO',
    # Septiembre
    'SEPTIEMBRE': 'SEPTIEMBRE', 'SEP': 'SEPTIEMBRE', 'septiembre': 'SEPTIEMBRE', 'Septiembre': 'SEPTIEMBRE', '09': 'SEPTIEMBRE', 'September': 'SEPTIEMBRE',
    # Octubre
    'OCTUBRE': 'OCTUBRE', '10': 'OCTUBRE', 'Octubre': 'OCTUBRE', 'octubre': 'OCTUBRE', 'October': 'OCTUBRE', 'OCT': 'OCTUBRE',
    # Noviembre
    'NOVIEMBRE': 'NOVIEMBRE', 'NOV': 'NOVIEMBRE', 'Noviembre': 'NOVIEMBRE', 'November': 'NOVIEMBRE', 'noviembre': 'NOVIEMBRE', '11': 'NOVIEMBRE',
    # Diciembre
    'DICIEMBRE': 'DICIEMBRE', '12': 'DICIEMBRE', 'diciembre': 'DICIEMBRE', 'Diciembre': 'DICIEMBRE', 'DIC': 'DICIEMBRE', 'December': 'DICIEMBRE'
}

# Aplicar mapeo sobre la columna mes
activaciones_limpio['mes'] = activaciones_limpio['mes'].astype(str).str.strip().map(mapeo_meses)

# VERIFICACIÓN
print("===== VERIFICACIÓN: DUPLICADOS Y MES =====")
print("• Nulos en 'mes':", activaciones_limpio['mes'].isna().sum())
print("• Categorías únicas resultantes en 'mes':")
print(activaciones_limpio['mes'].unique())

===== VERIFICACIÓN: DUPLICADOS Y MES =====
• Nulos en 'mes': 0
• Categorías únicas resultantes en 'mes':
<StringArray>
[     'ENERO',    'FEBRERO',      'MARZO',      'ABRIL',       'MAYO',
      'JUNIO',      'JULIO',     'AGOSTO', 'SEPTIEMBRE',    'OCTUBRE',
  'NOVIEMBRE',  'DICIEMBRE']
Length: 12, dtype: str


**SOLUCIÓN FORMATO HORA EN HORA_SOLICITUD Y HORA_INTERVENCIÓN**
- Estandarizar a HH:MM:SS. Cambiar ./- por :

In [10]:
def estandarizar_hora_completa(val):
    if pd.isna(val) or str(val).strip() in ['', 'nan', 'None']:
        return None
    
    # 1. Limpieza de texto y reemplazo de puntos y guiones por dos puntos
    txt = str(val).strip().replace('-', ':').replace('.', ':')
    
    # 2. Dividir componentes (HH:MM:SS)
    partes = txt.split(':')
    
    try:
        if len(partes) == 3:
            h, m, s = partes
            return f"{int(h):02d}:{int(m):02d}:{int(s):02d}"
        elif len(partes) == 2:
            h, m = partes
            return f"{int(h):02d}:{int(m):02d}:00"
    except ValueError:
        return txt
    
    return txt

# Aplicar a ambas columnas
activaciones_limpio['hora_solicitud'] = activaciones_limpio['hora_solicitud'].apply(estandarizar_hora_completa)
activaciones_limpio['hora_intervencion'] = activaciones_limpio['hora_intervencion'].apply(estandarizar_hora_completa)

# VERIFICACIÓN DEFINITIVA
print("===== VERIFICACIÓN DEFINITIVA: FORMATO DE HORAS =====")
patron_estandar = r'^\d{2}:\d{2}:\d{2}$'

for col in ['hora_solicitud', 'hora_intervencion']:
    horas_str = activaciones_limpio[col].dropna()
    cumplen_patron = horas_str.str.match(patron_estandar).sum()
    
    print(f"--- Columna '{col}' ---")
    print(f"• Total no nulos: {len(horas_str)}")
    print(f"• Cumplen formato estricto HH:MM:SS: {cumplen_patron} de {len(horas_str)}")
    print(f"• Nulos mantenidos intactos: {activaciones_limpio[col].isna().sum()}")

===== VERIFICACIÓN DEFINITIVA: FORMATO DE HORAS =====
--- Columna 'hora_solicitud' ---
• Total no nulos: 151258
• Cumplen formato estricto HH:MM:SS: 151258 de 151258
• Nulos mantenidos intactos: 1586
--- Columna 'hora_intervencion' ---
• Total no nulos: 138807
• Cumplen formato estricto HH:MM:SS: 138807 de 138807
• Nulos mantenidos intactos: 14037


**SOLUCIÓN OUTLIERS EDAD**
- Convertir a Nan aquellos fuera del rango [0-115]

In [12]:
# Convertir a NaN los valores de paciente_edad menores a 0 o mayores a 115
activaciones_limpio['paciente_edad'] = activaciones_limpio['paciente_edad'].apply(
    lambda x: np.nan if (pd.isna(x) or x < 0 or x > 115) else x
)

# VERIFICACIÓN DEFINITIVA DE EDAD
print("===== VERIFICACIÓN: COLUMNA PACIENTE_EDAD =====")
print("• Mínimo de edad:", activaciones_limpio['paciente_edad'].min())
print("• Máximo de edad:", activaciones_limpio['paciente_edad'].max())
print("\n• Total de valores nulos (NaN) en 'paciente_edad':", activaciones_limpio['paciente_edad'].isna().sum())
print("• Conteo de registros fuera del rango [0, 115]:", 
      ((activaciones_limpio['paciente_edad'] < 0) | (activaciones_limpio['paciente_edad'] > 115)).sum())

===== VERIFICACIÓN: COLUMNA PACIENTE_EDAD =====
• Mínimo de edad: 0.0
• Máximo de edad: 99.0

• Total de valores nulos (NaN) en 'paciente_edad': 950
• Conteo de registros fuera del rango [0, 115]: 0


**SOLUCIÓN MAPEO DISTRITO HOSPITAL Y PRIORIDAD**
- Estandarizar todos los textos a mayúsculas, eliminar tildes y unificar categorías equivalentes.

In [ ]:
#primero visualizamos todas las distintas etiquetas existentes
cols_a_inspeccionar = ['distrito', 'hospital', 'prioridad']

for col in cols_a_inspeccionar:
    print(f"===== VALORES ÚNICOS EN '{col}' (Total Nulos: {activaciones_limpio[col].isna().sum()}) =====")
    # Obtener valores únicos convertidos a lista
    unicos = activaciones_limpio[col].dropna().unique().tolist()
    print(unicos)
    print("\n")

===== VALORES ÚNICOS EN 'distrito' (Total Nulos: 840) =====
['FUENCARRAL', 'CENTRO', 'LATINA', 'CIUDAD LINEAL', 'CHAMBERI', 'SAN BLAS', 'RETIRO', 'TETUÁN', 'MONCLOA', 'VALLECAS PTE.', 'VICALVARO', 'ARGANZUELA', 'SALAMANCA', 'USERA', 'TETUAN', 'CARABANCHEL', 'MORATALAZ', 'HORTALEZA', 'VILLA DE VALLECAS', 'CHAMBERY', 'CHAMARTIN', 'VILLAVERDE', 'ARGANZELA', 'BARAJAS', 'Vallecas', 'LATNA', 'S. BLAS', 'barajas', 'P. VALLECAS', 'CENTO', 'TETAN', 'villaverde', 'CTR', 'arganzuela', 'FUERA TERMINO MUNICIPAL', 'Moncloa', 'USRA', 'Arganzuela', 'centro', 'Centro', 'usera', 'CDAD LINEAL', 'fuencarral', 'vicalvaro', 'Carabanchel', 'Tetuán', 'C. LINEAL', 'MONCLOA-ARAVACA', 'CRABANCHEL', 'Latina', 'San Blas', 'HORTLEZA', 'Hortaleza', 'PUENTE DE VALLECAS', 'Retiro', 'chamberi', 'FUENCARRAL-EL PARDO', 'tetuan', 'Ciudad Lineal', 'Barajas', 'CHAMBERÍ', 'Chamberí', 'salamanca', 'Salamanca', 'Usera', 'moncloa', 'retiro', 'Moratalaz', 'Villaverde', 'MORATALA', 'Fuencarral', 'carabanchel', 'SAN BLAS-CANILLEJA

In [16]:
# 1. FUNCIÓN DE LIMPIEZA DE TEXTO (Mayúsculas, sin tildes y sin espacios sobrantes)
def normalizar_texto(val):
    if pd.isna(val) or str(val).strip() in ['', 'nan', 'None']:
        return None
    
    texto = str(val).strip().upper()
    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto) 
        if unicodedata.category(c) != 'Mn'
    )
    return texto

# Aplicar normalización básica inicial
activaciones_limpio['distrito'] = activaciones_limpio['distrito'].apply(normalizar_texto)
activaciones_limpio['hospital'] = activaciones_limpio['hospital'].apply(normalizar_texto)
activaciones_limpio['prioridad'] = activaciones_limpio['prioridad'].apply(normalizar_texto)


# 2. MAPEO COMPLETO DE DISTRITOS
mapeo_distritos = {
    # Arganzuela
    'ARGANZUELA': 'ARGANZUELA', 'ARGANZELA': 'ARGANZUELA',
    # Barajas
    'BARAJAS': 'BARAJAS', 'BARJAS': 'BARAJAS',
    # Carabanchel
    'CARABANCHEL': 'CARABANCHEL', 'CRABANCHEL': 'CARABANCHEL',
    # Centro
    'CENTRO': 'CENTRO', 'CENTO': 'CENTRO', 'CTR': 'CENTRO',
    # Chamartín
    'CHAMARTIN': 'CHAMARTIN',
    # Chamberí
    'CHAMBERI': 'CHAMBERI', 'CHAMBERY': 'CHAMBERI',
    # Ciudad Lineal
    'CIUDAD LINEAL': 'CIUDAD LINEAL', 'CDAD LINEAL': 'CIUDAD LINEAL', 'C. LINEAL': 'CIUDAD LINEAL',
    # Fuencarral - El Pardo
    'FUENCARRAL': 'FUENCARRAL-EL PARDO', 'FUENCARRAL-EL PARDO': 'FUENCARRAL-EL PARDO',
    # Hortaleza
    'HORTALEZA': 'HORTALEZA', 'HORTLEZA': 'HORTALEZA',
    # Latina
    'LATINA': 'LATINA', 'LATNA': 'LATINA', 'LA LATINA': 'LATINA',
    # Moncloa - Aravaca
    'MONCLOA': 'MONCLOA-ARAVACA', 'MONCLOA-ARAVACA': 'MONCLOA-ARAVACA',
    # Moratalaz
    'MORATALAZ': 'MORATALAZ', 'MORATALA': 'MORATALAZ',
    # Puente de Vallecas
    'VALLECAS PTE.': 'PUENTE DE VALLECAS', 'P. VALLECAS': 'PUENTE DE VALLECAS', 
    'PUENTE DE VALLECAS': 'PUENTE DE VALLECAS', 'VALLECAS': 'PUENTE DE VALLECAS',
    # Retiro
    'RETIRO': 'RETIRO', 'EL RETIRO': 'RETIRO', 'RTIRO': 'RETIRO',
    # Salamanca
    'SALAMANCA': 'SALAMANCA', 'SALMANCA': 'SALAMANCA',
    # San Blas - Canillejas
    'SAN BLAS': 'SAN BLAS-CANILLEJAS', 'S. BLAS': 'SAN BLAS-CANILLEJAS', 'SAN BLAS-CANILLEJAS': 'SAN BLAS-CANILLEJAS',
    # Tetuán
    'TETUAN': 'TETUAN', 'TETAN': 'TETUAN',
    # Usera
    'USERA': 'USERA', 'USRA': 'USERA',
    # Vicálvaro
    'VICALVARO': 'VICALVARO',
    # Villa de Vallecas
    'VILLA DE VALLECAS': 'VILLA DE VALLECAS',
    # Villaverde
    'VILLAVERDE': 'VILLAVERDE', 'VILLA VERDE': 'VILLAVERDE',
    # Externe / Otros
    'FUERA TERMINO MUNICIPAL': 'FUERA DE TERMINO', 'POZUELO': 'FUERA DE TERMINO', 
    'C.A.M.': 'FUERA DE TERMINO', 'LEGANES': 'FUERA DE TERMINO',
    'CARRETERAS Y CIRCUNVALACIONES': 'CARRETERAS Y CIRCUNVALACIONES'
}

activaciones_limpio['distrito'] = activaciones_limpio['distrito'].replace(mapeo_distritos)


# 3. MAPEO COMPLETO DE HOSPITALES
mapeo_hospitales = {
    # La Paz
    'HULP': 'HOSPITAL LA PAZ', 'LA PAZ': 'HOSPITAL LA PAZ', 'H. LA PAZ': 'HOSPITAL LA PAZ', 'HOSPITAL LA PAZ': 'HOSPITAL LA PAZ',
    # Clínico San Carlos
    'CLINICO SAN CARLOS': 'HOSPITAL CLINICO SAN CARLOS', 'H. CLINICO': 'HOSPITAL CLINICO SAN CARLOS', 
    'SAN CARLOS': 'HOSPITAL CLINICO SAN CARLOS', 'HCSC': 'HOSPITAL CLINICO SAN CARLOS',
    # Fundación Jiménez Díaz
    'CONCEPCION (FUND. J. DIAZ)': 'HOSPITAL FUNDACION JIMENEZ DIAZ',
    # Gregorio Marañón
    'HGUGM': 'HOSPITAL GREGORIO MARANON', 'GREGORIO MARANON': 'HOSPITAL GREGORIO MARANON', 
    'MARANON': 'HOSPITAL GREGORIO MARANON', 'H. GREGORIO MARANON': 'HOSPITAL GREGORIO MARANON',
    # Ramón y Cajal
    'RAMON CAJAL': 'HOSPITAL RAMON Y CAJAL', 'RAMON Y CAJAL': 'HOSPITAL RAMON Y CAJAL', 
    'RYC': 'HOSPITAL RAMON Y CAJAL', 'H. RAMON Y CAJAL': 'HOSPITAL RAMON Y CAJAL',
    # La Princesa
    'LA PRINCESA': 'HOSPITAL LA PRINCESA',
    # 12 de Octubre
    'DOCE DE OCTUBRE': 'HOSPITAL DOCE DE OCTUBRE',
    # Infanta Leonor
    'INFANTA LEONOR': 'HOSPITAL INFANTA LEONOR',
    # Central de la Defensa / Gomez Ulla
    'CENTRAL DE LA DEFENSA': 'HOSPITAL CENTRAL DE LA DEFENSA GOMEZ ULLA',
    # Niño Jesús
    'NINO JESUS (INFANTIL)': 'HOSPITAL NINO JESUS',
    # Moncloa
    'MONCLOA (ASISA)': 'HOSPITAL MONCLOA',
    # Puerta de Hierro
    'PUERTA DE HIERRO': 'HOSPITAL PUERTA DE HIERRO',
    # Príncipe de Asturias
    'ALCALA DE HENARES (PPE. DE ASTURIAS)': 'HOSPITAL PRINCIPE DE ASTURIAS',
    # Getafe
    'GETAFE': 'HOSPITAL DE GETAFE',
    # Santa Cristina
    'SANTA CRISTINA': 'HOSPITAL SANTA CRISTINA',
    # Rodríguez Lafora
    'DOCTOR RODRIGUEZ LAFORA': 'HOSPITAL DOCTOR RODRIGUEZ LAFORA',
    # Genéricos / Municipales / Militares
    'OTROS CENTROS PUBLICOS': 'OTROS CENTROS PUBLICOS',
    'MILITARES (AIRE,POLIC. NAVAL, OTROS...)': 'CENTROS MILITARES',
    'CENTROS MUNICIPALES (ASIST. Y  ALBERGUES)': 'CENTROS MUNICIPALES',
    'CENTROS MUNICIPALES (ASIST. Y ALBERGUES)': 'CENTROS MUNICIPALES'
}

activaciones_limpio['hospital'] = activaciones_limpio['hospital'].replace(mapeo_hospitales)


# 4. MAPEO COMPLETO DE PRIORIDAD (1 -> ALTA, 2 -> MEDIA, 3 -> BAJA)
mapeo_prioridad = {
    'ALTA': 'ALTA', 'URGENTE': 'ALTA', '1': 'ALTA',
    'MEDIA': 'MEDIA', 'NORMAL': 'MEDIA', '2': 'MEDIA',
    'BAJA': 'BAJA', '3': 'BAJA'
}

activaciones_limpio['prioridad'] = activaciones_limpio['prioridad'].map(mapeo_prioridad)


# VERIFICACIÓN DEFINITIVA
print("===== VERIFICACIÓN: CATEGÓRICAS HOMOGENEIZADAS =====")

print("\n--- Columna 'distrito' ---")
print("• Categorías únicas resultantes:", activaciones_limpio['distrito'].nunique())
print("• Lista de distritos estandarizados:")
print(sorted(activaciones_limpio['distrito'].dropna().unique().tolist()))

print("\n--- Columna 'hospital' ---")
print("• Categorías únicas resultantes:", activaciones_limpio['hospital'].nunique())
print("• Lista de hospitales estandarizados:")
print(sorted(activaciones_limpio['hospital'].dropna().unique().tolist()))

print("\n--- Columna 'prioridad' ---")
print("• Categorías unificadas y conteo:")
print(activaciones_limpio['prioridad'].value_counts(dropna=False))

===== VERIFICACIÓN: CATEGÓRICAS HOMOGENEIZADAS =====

--- Columna 'distrito' ---
• Categorías únicas resultantes: 23
• Lista de distritos estandarizados:
['ARGANZUELA', 'BARAJAS', 'CARABANCHEL', 'CARRETERAS Y CIRCUNVALACIONES', 'CENTRO', 'CHAMARTIN', 'CHAMBERI', 'CIUDAD LINEAL', 'FUENCARRAL-EL PARDO', 'FUERA DE TERMINO', 'HORTALEZA', 'LATINA', 'MONCLOA-ARAVACA', 'MORATALAZ', 'PUENTE DE VALLECAS', 'RETIRO', 'SALAMANCA', 'SAN BLAS-CANILLEJAS', 'TETUAN', 'USERA', 'VICALVARO', 'VILLA DE VALLECAS', 'VILLAVERDE']

--- Columna 'hospital' ---
• Categorías únicas resultantes: 19
• Lista de hospitales estandarizados:
['CENTROS MILITARES', 'CENTROS MUNICIPALES', 'HOSPITAL CENTRAL DE LA DEFENSA GOMEZ ULLA', 'HOSPITAL CLINICO SAN CARLOS', 'HOSPITAL DE GETAFE', 'HOSPITAL DOCE DE OCTUBRE', 'HOSPITAL DOCTOR RODRIGUEZ LAFORA', 'HOSPITAL FUNDACION JIMENEZ DIAZ', 'HOSPITAL GREGORIO MARANON', 'HOSPITAL INFANTA LEONOR', 'HOSPITAL LA PAZ', 'HOSPITAL LA PRINCESA', 'HOSPITAL MONCLOA', 'HOSPITAL NINO JESUS', '

**SOLUCIÓN HOMOGENEIZACIÓN TIPO_EMERGENCIA**


In [18]:
# Función para quitar tildes y pasar a mayúsculas
def limpiar_emergencia(val):
    if pd.isna(val) or str(val).strip() in ['', 'nan', 'None']:
        return None
    
    texto = str(val).strip().upper()
    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto) 
        if unicodedata.category(c) != 'Mn'
    )
    return texto

# Aplicar a tipo_emergencia
activaciones_limpio['tipo_emergencia'] = activaciones_limpio['tipo_emergencia'].apply(limpiar_emergencia)

# VERIFICACIÓN
print("===== VERIFICACIÓN: TIPO_EMERGENCIA =====")
print("• Total de nulos en 'tipo_emergencia':", activaciones_limpio['tipo_emergencia'].isna().sum())
print("• Número de categorías únicas tras normalizar:", activaciones_limpio['tipo_emergencia'].nunique())
print("\nPrimeras 15 categorías resultantes:")
print(activaciones_limpio['tipo_emergencia'].dropna().unique()[:15])

===== VERIFICACIÓN: TIPO_EMERGENCIA =====
• Total de nulos en 'tipo_emergencia': 588
• Número de categorías únicas tras normalizar: 103

Primeras 15 categorías resultantes:
<StringArray>
[         'INTOXICACION ETILICA',      'AGRESION SIN ESPECIFICAR',
            'APERTURA DE PUERTA',      'PATOLOGIA CARDIOVASCULAR',
                       'HERIDAS',        'PATOLOGIA ANAFILACTICA',
      'CONVULSION Y/O EPILEPSIA',        'PATOLOGIA RESPIRATORIA',
                      'INCENDIO',    'CERTIFICACION PSIQUIATRICO',
            'CASUAL: CAIDA, ETC', 'ACCIDENTE MENOS DE 3 VICTIMAS',
                         'OTROS',                  'ORDEN MEDICA',
       'PATOLOGIA PSICOSOMATICA']
Length: 15, dtype: str


**TRATAMIENTO DE NULOS**
- Decidimos tratar los nulos de hospital, paciente_sexo, tipo_emergencia, distrito y prioridad como datos faltantes. Así pues cambiaremos esos nulos por la etiqueta NO_CONSTA

In [19]:
# 1. Imputación en 'hospital'
activaciones_limpio['hospital'] = activaciones_limpio['hospital'].fillna('SIN TRASLADO / NO CONSTA')

# 2. Imputación en 'paciente_sexo'
activaciones_limpio['paciente_sexo'] = activaciones_limpio['paciente_sexo'].fillna('NO CONSTA')

# 3. Imputación en 'tipo_emergencia' y 'distrito' (si se desea eliminar nulos por completo en categóricas)
activaciones_limpio['tipo_emergencia'] = activaciones_limpio['tipo_emergencia'].fillna('NO CONSTA')
activaciones_limpio['distrito'] = activaciones_limpio['distrito'].fillna('NO CONSTA')
activaciones_limpio['prioridad'] = activaciones_limpio['prioridad'].fillna('NO CONSTA')

# VERIFICACIÓN DE RELLENO
print("===== VERIFICACIÓN TRAS IMPUTACIÓN DE CATEGÓRICAS =====")
print(activaciones_limpio[['hospital', 'paciente_sexo', 'tipo_emergencia', 'distrito', 'prioridad']].isna().sum())

===== VERIFICACIÓN TRAS IMPUTACIÓN DE CATEGÓRICAS =====
hospital           0
paciente_sexo      0
tipo_emergencia    0
distrito           0
prioridad          0
dtype: int64


**ANÁLISIS DE DUPLICADOS**

In [21]:
# Inspeccionar filas duplicadas exactas (ordenadas para ver copias juntas)
duplicados_df = activaciones_limpio[activaciones_limpio.duplicated(keep=False)].sort_values(
    by=['activacion_id', 'hora_solicitud']
)

print(f"Total de filas involucradas en duplicación: {len(duplicados_df)}")
print("\nMuestra de registros idénticos:")
display(duplicados_df.head(10))

Total de filas involucradas en duplicación: 1476

Muestra de registros idénticos:


,activacion_id,año,mes,hora_solicitud,hora_intervencion,tipo_emergencia,distrito,hospital,unidad_id,prioridad,paciente_edad,paciente_sexo
205,206,2024,ENERO,07:29:32,07:41:43,AGRESION SIN ESPECIFICAR,TETUAN,SIN TRASLADO / NO CONSTA,49,MEDIA,11.0,M
152329,206,2024,ENERO,07:29:32,07:41:43,AGRESION SIN ESPECIFICAR,TETUAN,SIN TRASLADO / NO CONSTA,49,MEDIA,11.0,M
300,301,2024,ENERO,14:29:26,14:32:31,INCONSCIENTE SIN FILIAR CAUSA,CENTRO,SIN TRASLADO / NO CONSTA,36,BAJA,89.0,H
152321,301,2024,ENERO,14:29:26,14:32:31,INCONSCIENTE SIN FILIAR CAUSA,CENTRO,SIN TRASLADO / NO CONSTA,36,BAJA,89.0,H
406,407,2024,ENERO,19:17:49,19:25:20,PATOLOGIA CARDIOVASCULAR,CHAMBERI,SIN TRASLADO / NO CONSTA,50,MEDIA,5.0,M
152019,407,2024,ENERO,19:17:49,19:25:20,PATOLOGIA CARDIOVASCULAR,CHAMBERI,SIN TRASLADO / NO CONSTA,50,MEDIA,5.0,M
409,410,2024,ENERO,19:41:59,19:45:52,"CASUAL: CAIDA, ETC",MONCLOA-ARAVACA,HOSPITAL FUNDACION JIMENEZ DIAZ,13,MEDIA,5.0,M
151730,410,2024,ENERO,19:41:59,19:45:52,"CASUAL: CAIDA, ETC",MONCLOA-ARAVACA,HOSPITAL FUNDACION JIMENEZ DIAZ,13,MEDIA,5.0,M
635,636,2024,ENERO,13:41:38,13:49:01,"CASUAL: CAIDA, ETC",SALAMANCA,SIN TRASLADO / NO CONSTA,35,MEDIA,55.0,H
151987,636,2024,ENERO,13:41:38,13:49:01,"CASUAL: CAIDA, ETC",SALAMANCA,SIN TRASLADO / NO CONSTA,35,MEDIA,55.0,H
